# Day 06 - Building a Preprocessing Pipeline

So far we have been cleaning data manually step by step.
That works for exploration but it has a problem:

When we get new data (like test data), we have to manually repeat all those steps.
If we forget one step, the model breaks.

A Pipeline solves this. It puts all cleaning steps in one object.
You fit it once on training data, then use it on any new data automatically.

Today we will:
- Learn what a sklearn Pipeline is
- Build a simple pipeline for our data
- Test that it works correctly

## Step 1 - Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## Step 2 - Load the raw data again

We start from the original raw file.
The pipeline will do the cleaning for us this time.

In [3]:
df = pd.read_csv('cs-training.csv', index_col=0)
print('Shape:', df.shape)

Shape: (150000, 11)


In [4]:
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [25]:
df.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

## Step 3 - Separate features and target

X = all input columns (what we feed the model)
y = target column (what we want to predict)

In [5]:
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

In [6]:
print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (150000, 10)
y shape: (150000,)


In [7]:
# How many missing values are in X right now?
X.isnull().sum()

RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

## Step 4 - Add the flag column before pipeline

We learned on Day 02 that missing income is informative.
We need to add the flag column BEFORE we fill the missing values.
We do this manually before the pipeline.

In [8]:
X['MonthlyIncome_was_missing'] = X['MonthlyIncome'].isnull().astype(int)

In [9]:
# Check the new column
X['MonthlyIncome_was_missing'].value_counts()

MonthlyIncome_was_missing
0    120269
1     29731
Name: count, dtype: int64

## Step 5 - Also remove bad ages before pipeline

We remove impossible rows before the pipeline too.
The pipeline handles filling and scaling, not row removal.

In [27]:
X[X['age'] > 100]['age'].value_counts()

age
101    3
103    3
102    3
109    2
107    1
105    1
Name: count, dtype: int64

In [10]:
# Remove rows where age < 18
bad_age_rows = X[X['age'] < 18].index
print('Rows to remove:', len(bad_age_rows))

X = X.drop(index=bad_age_rows)
y = y.drop(index=bad_age_rows)

print('X shape after removing bad ages:', X.shape)


Rows to remove: 1
X shape after removing bad ages: (149999, 11)


## Step 6 - Cap the outliers before pipeline

In [28]:
X['RevolvingUtilizationOfUnsecuredLines'] = X['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.5)

In [29]:
X['DebtRatio'] = X['DebtRatio'].clip(upper=X['DebtRatio'].quantile(0.99))

In [30]:
X['MonthlyIncome'] = X['MonthlyIncome'].clip(upper=X['MonthlyIncome'].quantile(0.99))

In [31]:
for col in ['NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse']:
    X[col] = X[col].clip(upper=10)

## Step 7 - Build the Pipeline

Now we build the pipeline with two steps:
1. Imputer - fills remaining missing values with median
2. Scaler - makes all columns have similar scale (important for some models)

Note: Scaling means making all numbers roughly between -3 and +3.
Without scaling, a column with values in thousands can dominate a column with values 0-1.

In [15]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [16]:
# Look at what we built
print(pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


## Step 8 - Fit and transform the data

fit_transform does two things:
1. fit - it learns the median and scaling values FROM the data
2. transform - it applies those values to the data

In [17]:
X_processed = pipeline.fit_transform(X)

In [18]:
print('Shape after pipeline:', X_processed.shape)

Shape after pipeline: (149999, 11)


In [19]:
# Convert back to dataframe so it is easy to read
X_processed_df = pd.DataFrame(X_processed, columns=X.columns)
X_processed_df.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,MonthlyIncome_was_missing
0,1.235840,-0.493902,2.145664,-0.348138,0.754673,0.883651,-0.169825,4.409548,-0.154702,1.140539,-0.497198
1,1.768499,-0.832398,-0.325201,-0.348889,-0.908205,-0.865298,-0.169825,-0.901276,-0.154702,0.237210,-0.497198
2,0.934839,-0.967796,0.910231,-0.348929,-0.795476,-1.253953,1.403786,-0.901276,-0.154702,-0.666119,-0.497198
3,-0.248490,-1.509389,-0.325201,-0.348983,-0.729675,-0.670970,-0.169825,-0.901276,-0.154702,-0.666119,-0.497198
4,1.629324,-0.223106,0.910231,-0.348996,4.804751,-0.282315,-0.169825,-0.016139,-0.154702,-0.666119,-0.497198


## Step 9 - Check that missing values are gone

In [20]:
print('Missing values after pipeline:')
print(X_processed_df.isnull().sum())

Missing values after pipeline:
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
MonthlyIncome_was_missing               0
dtype: int64


## Step 10 - Check what scaling did

In [21]:
# Before scaling - original age values
print('Age BEFORE scaling:')
print('Min:', X['age'].min(), '  Max:', X['age'].max(), '  Mean:', round(X['age'].mean(), 2))

Age BEFORE scaling:
Min: 21   Max: 109   Mean: 52.3


In [22]:
# After scaling - age values are now small numbers
print('Age AFTER scaling:')
print('Min:', round(X_processed_df['age'].min(), 2), '  Max:', round(X_processed_df['age'].max(), 2), '  Mean:', round(X_processed_df['age'].mean(), 2))

Age AFTER scaling:
Min: -2.12   Max: 3.84   Mean: 0.0


The values are now centered around 0. All features are on the same scale now.
This helps models like Logistic Regression perform better.

## Step 11 - Save processed data

In [23]:
X_processed_df['SeriousDlqin2yrs'] = y.values
X_processed_df.to_csv('cs-training-day6-processed.csv', index=False)
print('Saved!')
print('Shape:', X_processed_df.shape)

Saved!
Shape: (149999, 12)


## My observations today (fill this yourself)

1. What is the difference between fit() and transform() : fit() learns parameters from data (mean, std, categories, etc.), while transform() applies those learned parameters to convert data. fit_transform() performs both operations together.
2. Why do we add the flag column BEFORE the pipeline, not inside it : We create the missing-value indicator before imputation so that the model can retain information about which values were originally missing. After imputation, that information is lost.
3. Why is scaling important? Which models need it : Scaling is important because many algorithms rely on distances or gradient optimization. Models such as Logistic Regression, KNN, SVM, and Neural Networks benefit from scaling, whereas tree-based models generally do not require it.
4. What happens if you use fit_transform() on test data :  (Hint: you should not)
   Using fit_transform() on test data causes data leakage because preprocessing parameters are learned from the test set. The test set should only be transformed using parameters learned from the training set to ensure a fair evaluation.

---
Tomorrow - Day 07: We create new features from existing ones